In [ ]:
import os 

os.chdir("..")

In [ ]:
from jp_imports import JPTrade
from jp_qcew import CleanQCEW
import polars as pl
import pymc as pm
import pandas as pd
from prprod_repl import ProdRepl
import arviz_base as azb
import arviz_plots as azp
azp.style.use("arviz-variat")


jt = JPTrade()
pr = ProdRepl()
cq = CleanQCEW(saving_dir="data/")

In [ ]:
gdf = pr.county_geom()[["name", "geoid"]]
gdf = pl.DataFrame(gdf)
df2 = gdf.with_columns(
        pl.col("name")
            .str.normalize("NFD")
            .str.replace_all(r"[\u0300-\u036f]", "")
            .str.to_lowercase()
            .str.replace_all(" ", "_")
        )
df2 = df2.to_pandas()
df2

In [ ]:
df = pr.data_qcew()
df = df.with_columns(
    name=(pl.col("phys_addr_city")
            .str.normalize("NFD")
            .str.replace_all(r"[\u0300-\u036f]", "")
            .str.to_lowercase()
            .str.replace_all("  ", "_")
            .str.replace_all(" ", "_")
        )
)
df_qcew = df.group_by(["year", "month", "sector_code"]).agg(pl.col("employment").sum())
new_df = df.group_by(["year", "month", "name", "sector_code"]).agg(pl.col("employment").sum()).to_pandas()

In [ ]:
temp = pl.read_parquet("data/raw/jp_data.parquet")
temp.with_columns(sec=pl.col("naics").str.slice(0,2)).filter(pl.col("sec")=="RE")

In [ ]:
temp = pl.read_parquet("data/raw/jp_data.parquet")
temp.select(sec=pl.col("naics").str.slice(0,2)).to_series().unique().sort().to_list()

In [ ]:
df_imports = jt.process_int_jp(level="naics", time_frame="monthly")
df_imports = df_imports.with_columns(
    sector_code=pl.col("naics").str.slice(0,2)
)#.rename({"qrt":"qtr"})
df_imports = df_imports.group_by(["year", "month", "sector_code"]).agg(pl.col("imports", "imports_qty", "exports","exports_qty","net_exports","net_qty").sum())
df_imports

In [ ]:
df = df_qcew.join(df_imports,on=["year", "month","sector_code"], how="inner", validate="1:1").to_pandas()
df

In [ ]:
import pymc as pm
import numpy as np
import pandas as pd
import pytensor.tensor as pt

In [ ]:
# 1. Create a continuous time index (Year + Month)
# This creates a sorted list of unique year-month combinations
time_coords = df[['year', 'month']].drop_duplicates().sort_values(['year', 'month'])
time_coords['time_idx'] = range(len(time_coords))

# Merge this index back into the main dataframe
df = df.merge(time_coords, on=['year', 'month'], how='left')

# 2. Preprocessing categorical indexes
df["sector_idx"], sectors = pd.factorize(df["sector_code"])

n_sectors = len(sectors)
n_periods = len(time_coords) # Total number of months in the timeseries

# Coordinate system for the model
coords = {
    "time": range(n_periods),
    "sector": sectors,
    "fourier_features": ["sin", "cos"]
}

In [ ]:
# Ensure obs_id is in your coords dictionary before the model block
coords["obs_id"] = np.arange(len(df))
month_scaled = 2 * np.pi * df["month"].values / 12
sin_t = np.sin(month_scaled)
cos_t = np.cos(month_scaled)

with pm.Model(coords=coords) as import_model:
    # --- 1. Data Containers ---
    # Use ConstantData for arrays. This converts them to PyTensor types automatically.
    s_idx = pm.Data("s_idx", df["sector_idx"].values, dims="obs_id")
    t_idx = pm.Data("t_idx", df["time_idx"].values, dims="obs_id")
    log_emp = pm.Data("log_emp", np.log(df["employment"].values + 1), dims="obs_id")

    # --- 2. Sector-Level Random Walk (Non-Centered) ---
    state_drift_mu = pm.Normal("state_drift_mu", mu=0, sigma=0.01)
    sector_drift = pm.Normal("sector_drift", mu=state_drift_mu, sigma=0.05, dims="sector")
    sigma_rw = pm.Exponential("sigma_rw", 2.0)
    
    z_innovations = pm.Normal("z_innovations", 0, 1, dims=("sector", "time"))
    
    init_val = np.log(df.groupby("sector_idx")["imports"].first().values + 1)
    init_import = pm.Normal("init_import", mu=init_val, sigma=0.5, dims="sector")
    
    # Path calculation
    rw_component = (z_innovations * sigma_rw).cumsum(axis=1)
    # Using np.arange is fine here as it broadcasts against the tensor
    drift_component = sector_drift[:, None] * np.arange(n_periods)
    
    sector_path = pm.Deterministic(
        "sector_path",
        init_import[:, None] + drift_component + rw_component,
        dims=("sector", "time")
    )

    # --- 3. Fourier Seasonality ---
    beta_fourier = pm.Normal("beta_fourier", mu=0, sigma=0.5, dims=("sector", "fourier_features"))
    
    # Corrected: Ensure sin_t and cos_t are also treated as data/tensors
    s_t = pm.Data("sin_t", sin_t, dims="obs_id")
    c_t = pm.Data("cos_t", cos_t, dims="obs_id")
    
    seasonality = (beta_fourier[s_idx, 0] * s_t + 
                   beta_fourier[s_idx, 1] * c_t)

    # --- 5. Employment Relationship ---
    beta_employment = pm.HalfNormal("beta_employment", sigma=1.0)

    # --- 6. Expected Value ---
    mu = pm.Deterministic(
        "mu",
        sector_path[s_idx, t_idx] + 
        (beta_employment * log_emp) + 
        seasonality,
        dims="obs_id"
    )

    # --- 7. Likelihood ---
    sigma_obs = pm.Exponential("sigma_obs", 1.0)
    nu = pm.Exponential("nu", 0.1) + 1 
    
    pm.StudentT(
        "obs",
        nu=nu,
        mu=mu,
        sigma=sigma_obs,
        observed=np.log(df["imports"].values + 1),
        dims="obs_id"
    )

    trace = pm.sample(1000, tune=1000, target_accept=0.95)

In [ ]:
import altair as alt
import arviz as az
import pandas as pd
import numpy as np

def plot_sector_import_forecast(trace, data, sector_name: str, time_coords_df: pd.DataFrame):
    """
    Plots the random walk path for a specific sector.
    time_coords_df should be the DataFrame you created with ['year', 'month', 'time_idx']
    """
    # 1. Prep Time labels from the DataFrame
    # Ensure we have a proper datetime objects for Altair
    time_labels = time_coords_df.copy().sort_values("time_idx")
    time_labels['date'] = pd.to_datetime(
        time_labels['year'].astype(str) + '-' + time_labels['month'].astype(str) + '-01'
    )
    
    # 2. Extract Posterior for the specific sector
    # .sel(sector=...) works because we used 'coords' in the PyMC model
    log_paths = trace.posterior["sector_path"].sel(sector=sector_name).values
    
    # Reshape and back-transform: exp(x) - 1
    # This converts the de-seasoned latent trend back to normal units
    path_samples = np.exp(log_paths.reshape(-1, len(time_labels))) - 1

    # 3. Calculate Stats
    median_path = np.median(path_samples, axis=0)
    hdi_95 = az.hdi(path_samples, hdi_prob=0.95)
    hdi_50 = az.hdi(path_samples, hdi_prob=0.50)

    # 4. Create Tidy DataFrame for Altair
    plot_df = pd.DataFrame({
        "date": time_labels['date'],
        "median": median_path,
        "hdi_95_low": hdi_95[:, 0],
        "hdi_95_high": hdi_95[:, 1],
        "hdi_50_low": hdi_50[:, 0],
        "hdi_50_high": hdi_50[:, 1],
    })

    # Prepare observed points
    obs_data = data[data["sector_code"] == sector_name].copy()
    obs_data['date'] = pd.to_datetime(
        obs_data['year'].astype(str) + '-' + obs_data['month'].astype(str) + '-01'
    )

    # 5. Define Altair Layers
    base = alt.Chart(plot_df).encode(x=alt.X("date:T", title="Date"))

    # Uncertainty Bands
    band_95 = base.mark_area(opacity=0.1, color="steelblue").encode(
        y=alt.Y("hdi_95_low:Q", title="Import Value"),
        y2="hdi_95_high:Q"
    )
    band_50 = base.mark_area(opacity=0.3, color="steelblue").encode(
        y="hdi_50_low:Q",
        y2="hdi_50_high:Q"
    )

    # Latent Trend Line
    line = base.mark_line(strokeWidth=3, color="#1f77b4").encode(
        y="median:Q"
    )

    # Raw Observed Points
    points = alt.Chart(obs_data).mark_point(
        size=50, filled=True, color="black", opacity=0.6
    ).encode(
        x="date:T",
        y="imports:Q",
        tooltip=["year", "month", "imports"]
    )

    # Final Composite
    chart = (
        alt.layer(band_95, band_50, line, points)
        .properties(
            title={
                "text": f"Sector {sector_name}: Latent Import Trend",
                "subtitle": "Blue line shows de-seasoned random walk; Black dots are raw data",
                "anchor": "start",
            },
            width=700,
            height=400,
        )
        .configure_axis(gridDash=[3, 3], gridColor="#eeeeee")
        .interactive()
    )

    return chart

# --- HOW TO CALL IT ---
# Pass your original 'time_coords' DataFrame instead of range()
chart = plot_sector_import_forecast(trace, df, "21", time_coords)
chart.show()

In [ ]:
# 1. Map Sectors to the same indices used in training
# 'sectors' should be the list/index of sector names from your original coords
new_df["sector_idx"] = pd.Categorical(new_df["sector_code"], categories=sectors).codes

# 2. Map Year/Month to the same time_idx used in training
# time_coords is the dataframe with columns ['year', 'month', 'time_idx']
new_df = new_df.merge(time_coords[['year', 'month', 'time_idx']], on=['year', 'month'], how='left')

# 3. Calculate Fourier features for the specific months in new_df
new_month_scaled = 2 * np.pi * new_df["month"].values / 12
new_sin_t = np.sin(new_month_scaled)
new_cos_t = np.cos(new_month_scaled)

# 4. Log-transform employment
new_log_emp = np.log(new_df["employment"].values + 1)

In [ ]:
# 1. Check for missing indices after the merge
if new_df["time_idx"].isnull().any():
    print("Warning: Some rows in new_df don't have a matching time_idx. Dropping them.")
    new_df = new_df.dropna(subset=["time_idx", "sector_idx"])

# 2. Prepare the values directly from existing columns
# Make sure "employment" is the correct name of your column in new_df
new_s_idx = new_df["sector_idx"].values.astype("int32")
new_t_idx = new_df["time_idx"].values.astype("int32")

# Calculate log(emp + 1) and force float64
new_log_emp_values = np.log(new_df["employment"].values + 1).astype("float64")

# Re-calculate Fourier features for the new months if you haven't yet
new_month_scaled = 2 * np.pi * new_df["month"].values / 12
new_sin_t = np.sin(new_month_scaled).astype("float64")
new_cos_t = np.cos(new_month_scaled).astype("float64")

with import_model:
    # 1. Update the data as before
    pm.set_data(
        new_data={
            "s_idx": new_s_idx,
            "t_idx": new_t_idx,
            "log_emp": new_log_emp_values,
            "sin_t": new_sin_t,
            "cos_t": new_cos_t
        },
        coords={"obs_id": np.arange(len(new_df))}
    )
    
    # 2. Add 'predictions=True' to avoid coordinate conflicts with training data
    post_pred = pm.sample_posterior_predictive(
        trace, 
        var_names=["obs"], 
        predictions=True
    )

# 3. Extract from the 'predictions' group instead of 'posterior_predictive'
synthetic_log_mean = post_pred.predictions["obs"].mean(dim=["chain", "draw"]).values

# 4. Transform back
new_df["synthetic_imports"] = np.exp(synthetic_log_mean) - 1

In [ ]:
new_df